In [ ]:
import re
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

DATA_ROOT = "../../data"

df = pd.read_parquet(os.path.join(DATA_ROOT, "full_merged_data.parquet"))

df["real_text"] = df["real_text"].astype(str).apply(lambda x: x.strip())
df["generated_text"] = df["generated_text"].astype(str).apply(lambda x: x.strip())

real = df["real_text"].astype(str)
gen  = df["generated_text"].astype(str)

print(f"Total rows : {len(df):,}")
print(df.groupby(["platform", "model", "pipeline"]).size().rename("rows"))

In [ ]:
g1_html_entity  = real.str.contains(r"&[a-z]{2,6};", regex=True, na=False) | gen.str.contains(r"&[a-z]{2,6};", regex=True, na=False)
is_group1 = g1_html_entity

print("Group 1 per-flag rates:")
for name, flag in [("html_entity",  g1_html_entity),]:
    print(f"  {name:16s}: {flag.sum():>6,}  ({flag.mean()*100:.2f}%)")

In [ ]:
REAL_THR = 10
GEN_THR = 10
g2_real_short = rl < REAL_THR
g2_gen_short  = gl < GEN_THR
is_group2 = g2_real_short | g2_gen_short

print(f"real_text < {REAL_THR}   : {g2_real_short.sum():>6,}  ({g2_real_short.mean()*100:.2f}%)")
print(f"gen_text  < {GEN_THR}   : {g2_gen_short.sum():>6,}  ({g2_gen_short.mean()*100:.2f}%)")
print(f"overlap (both)         : {(g2_real_short & g2_gen_short).sum():>6,}")
print(f"Group 2 union          : {is_group2.sum():>6,}  ({is_group2.mean()*100:.2f}%)")

In [ ]:
df_final = df_w[~is_group2.values].reset_index(drop=True)
print(f"  Original              : {len(df):>8,}")
print(f"  After Group 1         : {len(df_w):>8,}  ({len(df_w)/len(df)*100:.1f}% retained)")
print(f"  After Group 1+2       : {len(df_final):>8,}  ({len(df_final)/len(df)*100:.1f}% retained)")

In [ ]:
out_path = os.path.join(DATA_ROOT, "full_merged_data_filtered.parquet")
df_final.to_parquet(out_path, index=False)
print(f"Saved: {len(df_final):,} rows, {out_path}")